In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
import re
import pprint

In [2]:
# Load environment variables
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = OpenAI()

# Define shared state
class ScrumProjectState(TypedDict):
    goal: str
    #CustomerProxy: Optional[str]
    ProductOwner: Optional[str]
    RequirementsEngineer: Optional[str]
    SystemEngineer: Optional[str]
    SoftwareDeveloper: Optional[str]
    TestEngineer: Optional[str]
    ScrumMaster: Optional[str]
    DocumentationEngineer: Optional[str]
    UIUXDesigner: Optional[str]


In [3]:
scrum_prompts = {
    "CustomerProxy": "You are a representative of the educational institution requesting a new web-based educational portal. "
    "You will act as a proxy for the customer and provide the initial user stories, stakeholder goals, and success criteria. "
    "Please describe the core problems faced by the institution, the key features expected (course management, assignments, grading, communication, etc.), "
    "and any constraints or priorities (budget, timeline, accessibility, etc.). Keep it realistic and focused on stakeholder needs.",

    "ProductOwner": "The educational institution has requested the creation of a web-based educational portal. It should support course management, assignment submission and grading, forums, document sharing, grade tracking, and communication features. As the ProductOwner, your role is to: - Relay this problem to other agents (RequirementsEngineer, SystemEngineer, SoftwareDeveloper, etc.). - Gather detailed outputs from each and maintain overall alignment with stakeholder goals. - Ensure estimates and productivity metrics are tracked. Start by sending messages to the RequirementsEngineer, SystemEngineer, SoftwareDeveloper, TestEngineer, ScrumMaster, DocumentationEngineer, and UIUXDesigner.",

    "RequirementsEngineer": "I’ve received the following problem statement from our educational institution: \"We want to build a web-based educational portal where students and faculty can manage courses, submit and grade assignments, participate in discussion forums, share documents, view grades, and communicate effectively.\" Please provide the following: 1. A detailed list of use-cases with unique names and numbers. 2. For each use-case, define both functional and non-functional requirements with unique identifiers. 3. Estimate your productivity (e.g., how many use-cases and requirements can you document per hour/day). 4. List all tasks involved in the requirements phase, including reviews and rework. 5. Provide the total estimated effort for the requirements phase (in hours/days).",

    "SystemEngineer": "Based on the requirements document provided, generate a complete system design plan. 1. Specify the architectural style you recommend (e.g., microservices, layered architecture). 2. Create a list of design artifacts (e.g., pages, modules, APIs, DB schema, diagrams). 3. Estimate your productivity (e.g., how many diagrams or components per day/hour). 4. Identify all design tasks, including review and iteration. 5. Provide a total effort estimate (in hours/days) for the design phase.",

    "SoftwareDeveloper": "Based on the finalized design and use-cases: 1. Break the major features into coding tasks and components. 2. Estimate code quantity per module (e.g., lines of code, functions, files). 3. Provide productivity estimates (e.g., how many lines/modules per hour/day). 4. List all development tasks: setup, implementation, review, testing, integration, refactoring. 5. Estimate time (in hours/days) per major feature. 6. Mention any dependencies (e.g., backend APIs before frontend integration).",

    "TestEngineer": "Using the system design and requirements: 1. Identify the number of test cases per use-case. 2. Classify test cases (unit, integration, functional, regression, etc.). 3. Provide productivity estimates (e.g., test cases authored/executed per hour/day). 4. List testing tasks (planning, writing, execution, bug logging, re-testing). 5. Estimate total testing effort (in hours/days) for all components.",

    "ScrumMaster": "Based on the outputs from all other agents, extract and summarize the actual estimated effort (in days) provided by each role. Then, generate a sprint plan that distributes the tasks accordingly. Assume each sprint is 2 weeks and each agent can work 40 hours per week.",

    "DocumentationEngineer": "Create documentation for the educational platform project: 1. Identify documents required: requirement specs, architecture docs, test plans, user manuals. 2. Estimate productivity (e.g., how many pages/sections you can write per day). 3. List documentation tasks (drafting, reviewing, formatting, versioning). 4. Provide effort estimates (in hours/days) for the documentation phase. 5. Suggest documentation tools or formats for efficient collaboration.",

    "UIUXDesigner": "Design the UI/UX for the educational portal: 1. Create wireframes and high-level UI mockups for main pages (dashboard, course pages, forums, assignment pages, etc.). 2. Consider responsive design for both desktop and mobile use. 3. List the tools you would use. 4. Provide a breakdown of tasks and estimated effort in hours or days for wireframing, feedback iteration, prototyping, and design finalization."
}

In [4]:
def run_agent(role, state: ScrumProjectState) -> ScrumProjectState:
    prompt = f"{scrum_prompts[role]}\nProject Goal: {state['goal']}"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2000
    )
    content = response.choices[0].message.content.strip()

    new_state = state.copy()
    new_state[role] = content  # <--- simpler and more direct
    return new_state

In [6]:
graph = StateGraph(ScrumProjectState)

roles = [
    "CustomerProxy", "ProductOwner", "RequirementsEngineer", "SystemEngineer",
    "SoftwareDeveloper", "TestEngineer", "DocumentationEngineer",
    "UIUXDesigner", "ScrumMaster"
]

# Add nodes using unique node names
for role in roles:
    node_name = f"{role}_node"  # prevent collision with state keys
    graph.add_node(node_name, lambda s, r=role: run_agent(r, s))

# Entry and flow
graph.set_entry_point("CustomerProxy_node")

# Chain the nodes in order
for i in range(len(roles) - 1):
    graph.add_edge(f"{roles[i]}_node", f"{roles[i+1]}_node")
graph.add_edge(f"{roles[-1]}_node", END)

flow = graph.compile()

In [7]:
# Initial state
goal = "Create an educational portal where students and teachers can manage courses, submit assignments, grade work, participate in forums, and share documents."
initial_state: ScrumProjectState = {
    "goal": goal,
    "product_backlog": None,
    "sprint_plan": None,
    "dev_tasks": None,
    "qa_plan": None,
    "uiux_plan": None,
    "requirements": None,
    "system_design": None,
     "summary": None
}

In [8]:
# Run pipeline
final_state = flow.invoke(initial_state)

# Display final output
pprint.pprint(final_state)


{'DocumentationEngineer': '# Documentation for Educational Platform Project\n'
                          '\n'
                          '## 1. Required Documents\n'
                          '\n'
                          '### Requirements Specifications Document (RSD)\n'
                          '- Define functional and non-functional '
                          'requirements\n'
                          '- User stories and use cases\n'
                          '- Stakeholder analysis\n'
                          '\n'
                          '### Architecture Documentation\n'
                          '- High-level architecture design\n'
                          '- Component diagrams\n'
                          '- System interface designs\n'
                          '- Technology stack used\n'
                          '\n'
                          '### Test Plans\n'
                          '- Test strategy (unit, integration, system, user '
                          'accept